# Student Academic Performance Analysis
## SAP-4000 Dataset - Complete EDA and Statistical Analysis

This notebook performs comprehensive analysis of student academic performance data.

**Dataset**: 4,000 student records from Spain's Título de Bachiller system

**Analysis includes**:
- Data loading and cleaning
- Exploratory Data Analysis (EDA)
- Statistical testing (t-tests, ANOVA, Chi-square)
- Multiple linear regression
- Data visualization
- Interpretation and insights

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind, f_oneway, pearsonr
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Data Loading

In [ ]:
# Load dataset
df = pd.read_csv("SAP-4000.csv")

print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)
df.head()

## 2. Data Quality Check

In [ ]:
# Check missing values
print("Missing Values:")
print(df.isnull().sum())

# Check duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Descriptive statistics
df.describe()

## 3. Data Cleaning & Feature Engineering

In [ ]:
# Create copy for cleaning
df_clean = df.copy()

# Fill missing Parent Education
mode_edu = df_clean['Parent Education'].mode()[0]
df_clean['Parent Education'].fillna(mode_edu, inplace=True)

# Feature Engineering
df_clean['Study_Category'] = pd.cut(df_clean['HoursStudied/Week'], 
                                    bins=[0, 5, 10, 15, 20], 
                                    labels=['Low', 'Medium', 'High', 'Very High'])

df_clean['Attendance_Category'] = pd.cut(df_clean['Attendance(%)'], 
                                         bins=[0, 60, 75, 90, 100], 
                                         labels=['Poor', 'Fair', 'Good', 'Excellent'])

df_clean['Score_Category'] = pd.cut(df_clean['Exam_Score'], 
                                  bins=[0, 50, 70, 85, 100], 
                                  labels=['Fail', 'Pass', 'Good', 'Excellent'])

df_clean['Study_Efficiency'] = df_clean['Exam_Score'] / (df_clean['HoursStudied/Week'] + 1)

print(f"Cleaned dataset shape: {df_clean.shape}")

## 4. Exploratory Data Analysis

In [ ]:
# Categorical distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender
df_clean['Gender'].value_counts().plot(kind='bar', ax=axes[0,0], color=['#FF6B9D', '#4ECDC4'])
axes[0,0].set_title('Gender Distribution')

# Tutoring
df_clean['Tutoring'].value_counts().plot(kind='bar', ax=axes[0,1], color=['#95E1D3', '#F38181'])
axes[0,1].set_title('Tutoring Distribution')

# Region
df_clean['Region'].value_counts().plot(kind='bar', ax=axes[1,0], color=['#AA96DA', '#FCBAD3'])
axes[1,0].set_title('Region Distribution')

# Parent Education
df_clean['Parent Education'].value_counts().plot(kind='bar', ax=axes[1,1], 
                                                   color=['#FFD93D', '#6BCB77', '#4D96FF', '#FF6B6B'])
axes[1,1].set_title('Parent Education Distribution')

plt.tight_layout()
plt.show()

## 5. Statistical Testing

In [ ]:
# T-test: Gender differences
male_scores = df_clean[df_clean['Gender'] == 'Male']['Exam_Score']
female_scores = df_clean[df_clean['Gender'] == 'Female']['Exam_Score']
t_stat, p_val = ttest_ind(male_scores, female_scores)

print("Gender Differences:")
print(f"Female mean: {female_scores.mean():.2f}")
print(f"Male mean: {male_scores.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.6f}")
print(f"Result: {'SIGNIFICANT' if p_val < 0.05 else 'Not significant'}")

In [ ]:
# T-test: Tutoring effect
tutor_yes = df_clean[df_clean['Tutoring'] == 'Yes']['Exam_Score']
tutor_no = df_clean[df_clean['Tutoring'] == 'No']['Exam_Score']
t_stat, p_val = ttest_ind(tutor_yes, tutor_no)

print("Tutoring Effect:")
print(f"With tutoring: {tutor_yes.mean():.2f}")
print(f"Without tutoring: {tutor_no.mean():.2f}")
print(f"Difference: +{tutor_yes.mean() - tutor_no.mean():.2f} points")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.6e}")
print(f"Result: {'SIGNIFICANT' if p_val < 0.05 else 'Not significant'}")

In [ ]:
# ANOVA: Parent Education
education_groups = [group['Exam_Score'].values for name, group in df_clean.groupby('Parent Education')]
f_stat, p_val = f_oneway(*education_groups)

print("Parent Education Effect (ANOVA):")
print(df_clean.groupby('Parent Education')['Exam_Score'].mean().round(2))
print(f"\nF-statistic: {f_stat:.4f}, p-value: {p_val:.6e}")
print(f"Result: {'SIGNIFICANT' if p_val < 0.05 else 'Not significant'}")

## 6. Multiple Linear Regression

In [ ]:
# Prepare data
df_reg = df_clean.copy()
df_reg['Gender_Male'] = (df_reg['Gender'] == 'Male').astype(int)
df_reg['Tutoring_Yes'] = (df_reg['Tutoring'] == 'Yes').astype(int)
df_reg['Region_Urban'] = (df_reg['Region'] == 'Urban').astype(int)

# One-hot encode Parent Education
edu_dummies = pd.get_dummies(df_reg['Parent Education'], prefix='Edu', drop_first=True)
df_reg = pd.concat([df_reg, edu_dummies], axis=1)

# Define features
features = ['HoursStudied/Week', 'Attendance(%)', 'Gender_Male', 
            'Tutoring_Yes', 'Region_Urban'] + list(edu_dummies.columns)

X = df_reg[features].astype(float)
y = df_reg['Exam_Score'].astype(float)
X_const = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X_const).fit()
print(model.summary())

## 7. Results Summary

In [ ]:
# Print key results
print("\n" + "="*60)
print("KEY FINDINGS")
print("="*60)

print(f"\n1. TUTORING EFFECT:")
print(f"   With tutoring: {tutor_yes.mean():.1f} points")
print(f"   Without tutoring: {tutor_no.mean():.1f} points")
print(f"   Difference: +{tutor_yes.mean() - tutor_no.mean():.1f} points (p<0.001)")

print(f"\n2. GENDER GAP:")
print(f"   Female: {female_scores.mean():.1f} points")
print(f"   Male: {male_scores.mean():.1f} points")
print(f"   Difference: +{female_scores.mean() - male_scores.mean():.1f} points (p<0.001)")

print(f"\n3. REGRESSION MODEL:")
print(f"   R-squared: {model.rsquared:.3f} (explains {model.rsquared*100:.1f}% of variance)")
print(f"   F-statistic: {model.fvalue:.2f} (p<0.001)")

print(f"\n4. STRONGEST PREDICTORS:")
coef_sorted = model.params.abs().sort_values(ascending=False)
for var in coef_sorted.head(5).index:
    if var != 'const':
        print(f"   {var}: {model.params[var]:+.3f}")

## 8. Save Results

In [ ]:
# Save files
df_clean.to_csv('SAP-4000_cleaned.csv', index=False)
print("✓ Saved: SAP-4000_cleaned.csv")

# Save regression results
regression_results = pd.DataFrame({
    'Variable': model.params.index,
    'Coefficient': model.params.values,
    'Std_Error': model.bse.values,
    'p_value': model.pvalues.values
})
regression_results.to_csv('regression_results.csv', index=False)
print("✓ Saved: regression_results.csv")

print("\nAnalysis complete!")